## Generalization Gap between Val & Test

In [1]:
import mlflow
import pandas as pd

In [2]:
METRICS = {
    'val_base_acoustic': mlflow.get_run('35fadeeec29c477186f66d32a4b27a86').data.metrics,
    'val_base_linguistic_ref': mlflow.get_run('6ab4d8e5e7cb41ae82168df17dc35571').data.metrics,
    'val_base_linguistic_trans': mlflow.get_run('438a002bce8b428cb26d38a264c8b53b').data.metrics,
    'val_base_fusion_ref': mlflow.get_run('bba6ccf53236443c9b70007916e99efe').data.metrics,
    'val_base_fusion_trans': mlflow.get_run('f88d1b03550e442490a529cd65fe7a6d').data.metrics,

    'test_base_acoustic': mlflow.get_run('1d8c7b79d1724d6c99f56c28aff811c4').data.metrics,
    'test_base_linguistic_ref': mlflow.get_run('d471f38ed3f54dc09f15436491d9ff3e').data.metrics,
    'test_base_linguistic_trans': mlflow.get_run('db62bee334f440edb92cad5673d3a43f').data.metrics,
    'test_base_fusion_ref': mlflow.get_run('62eb83d6e1e841598c1e26302ddd2a59').data.metrics,
    'test_base_fusion_trans': mlflow.get_run('bd8e9cb571754ff7ae099d5901bb6a70').data.metrics
}

In [3]:
metrics = {
    'train_bal_acc': [METRICS['val_base_acoustic']['bal_accuracy'], METRICS['val_base_linguistic_ref']['bal_accuracy'], METRICS['val_base_linguistic_trans']['bal_accuracy'], METRICS['val_base_fusion_ref']['bal_accuracy'], METRICS['val_base_fusion_trans']['bal_accuracy']],

    'val_bal_acc': [METRICS['val_base_acoustic']['best_val_ba'], METRICS['val_base_linguistic_ref']['best_val_ba'], METRICS['val_base_linguistic_trans']['best_val_ba'], METRICS['val_base_fusion_ref']['val_bal_acc'], METRICS['val_base_fusion_trans']['best_val_ba']],

    'test_bal_acc': [METRICS['test_base_acoustic']['balanced_accuracy'], METRICS['test_base_linguistic_ref']['balanced_accuracy'], METRICS['test_base_linguistic_trans']['balanced_accuracy'], METRICS['test_base_fusion_ref']['balanced_accuracy'], METRICS['test_base_fusion_trans']['balanced_accuracy']],

    'train_macro_f1': [METRICS['val_base_acoustic']['macro_f1'], METRICS['val_base_linguistic_ref']['macro_f1'], METRICS['val_base_linguistic_trans']['macro_f1'], METRICS['val_base_fusion_ref']['macro_f1'], METRICS['val_base_fusion_trans']['macro_f1']], 

    'val_macro_f1': [METRICS['val_base_acoustic']['best_val_mf1'], METRICS['val_base_linguistic_ref']['best_val_mf1'], METRICS['val_base_linguistic_trans']['best_val_mf1'], METRICS['val_base_fusion_ref']['best_val_mf1'], METRICS['val_base_fusion_trans']['best_val_mf1']],

    'test_macro_f1': [METRICS['test_base_acoustic']['macro_f1'], METRICS['test_base_linguistic_ref']['macro_f1'], METRICS['test_base_linguistic_trans']['macro_f1'], METRICS['test_base_fusion_ref']['macro_f1'], METRICS['test_base_fusion_trans']['macro_f1']]
}

cols = ['base acoustic', 'base linguistic ref', 'base linguistic trans', 'fusion ref', 'fusion trans']
comp_table = pd.DataFrame(metrics, cols)
comp_table

,train_bal_acc,val_bal_acc,test_bal_acc,train_macro_f1,val_macro_f1,test_macro_f1
base acoustic,0.695598,0.288908,0.189569,0.746823,0.276592,0.185007
base linguistic ref,0.595986,0.418092,0.407067,0.639201,0.416924,0.396583
base linguistic trans,0.545964,0.295208,0.252734,0.594627,0.308378,0.268208
fusion ref,0.818613,0.366847,0.373016,0.847953,0.403711,0.369989
fusion trans,0.789015,0.355467,0.269343,0.819986,0.349083,0.279410


In [4]:
def find_valtest_gap(val_metrics, test_metrics):
    bal_acc_gap = val_metrics['best_val_ba'] - test_metrics['balanced_accuracy']
    mf1_gap = val_metrics['best_val_mf1'] - test_metrics['macro_f1']

    print(f"balanced accuracy gap: {(bal_acc_gap* 100):.4f}%")
    print(f"macro f1 score gap: {(mf1_gap * 100):.4f}%")

In [5]:
def find_trainval_gap(train_metrics, val_metrics):
    bal_acc_gap = (train_metrics['bal_accuracy'] - val_metrics['best_val_ba']) * 100
    mf1_gap = (val_metrics['macro_f1'] - val_metrics['best_val_mf1']) * 100

    print(f"balanced accuracy gap: {bal_acc_gap:.4f}%")
    print(f"macro f1 score gap: {mf1_gap:.4f}%")

    if abs(bal_acc_gap) < 5:
        print("good generalization.")
    else:
        print("severe overfitting!!")

### Acoustic Only Baseline

In [6]:
find_trainval_gap(METRICS['val_base_acoustic'], METRICS['val_base_acoustic'])

balanced accuracy gap: 40.6691%
macro f1 score gap: 47.0231%
severe overfitting!!


In [7]:
find_valtest_gap(METRICS['val_base_acoustic'], METRICS['test_base_acoustic'])

balanced accuracy gap: 9.9338%
macro f1 score gap: 9.1586%


### Linguistic Only with Reference Transcripts Baseline

In [8]:
find_trainval_gap(METRICS['val_base_linguistic_ref'], METRICS['val_base_linguistic_ref'])

balanced accuracy gap: 17.7894%
macro f1 score gap: 22.2276%
severe overfitting!!


In [9]:
find_valtest_gap(METRICS['val_base_linguistic_ref'], METRICS['test_base_linguistic_ref'])

balanced accuracy gap: 1.1025%
macro f1 score gap: 2.0341%


### Linguistic Only with Whisper Transcripts Baseline

In [10]:
find_trainval_gap(METRICS['val_base_linguistic_trans'], METRICS['val_base_linguistic_trans'])

balanced accuracy gap: 25.0756%
macro f1 score gap: 28.6249%
severe overfitting!!


In [11]:
find_valtest_gap(METRICS['val_base_linguistic_trans'], METRICS['test_base_linguistic_trans'])

balanced accuracy gap: 4.2474%
macro f1 score gap: 4.0170%


### Fusion with Reference Transcripts Baseline

In [12]:
find_trainval_gap(METRICS['val_base_fusion_ref'], METRICS['val_base_fusion_ref'])

balanced accuracy gap: 41.0083%
macro f1 score gap: 44.4243%
severe overfitting!!


In [13]:
find_valtest_gap(METRICS['val_base_fusion_ref'], METRICS['test_base_fusion_ref'])

balanced accuracy gap: 3.5514%
macro f1 score gap: 3.3722%


### Fusion with Whisper Transcripts Baseline

In [14]:
find_trainval_gap(METRICS['val_base_fusion_trans'], METRICS['val_base_fusion_trans'])

balanced accuracy gap: 43.3548%
macro f1 score gap: 47.0903%
severe overfitting!!


In [15]:
find_valtest_gap(METRICS['val_base_linguistic_trans'], METRICS['test_base_linguistic_trans'])

balanced accuracy gap: 4.2474%
macro f1 score gap: 4.0170%
